In [1]:
# ============================================
# YOLO MODEL OPTIMIZATION
# Environment Setup
# ============================================

!nvidia-smi

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Sat Sep 26 13:59:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ============================================
# INSTALL OPTIMIZATION LIBRARIES
# ============================================

!pip install -q ultralytics==8.4.27 torch-pruning==1.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.2 MB/s eta 0:00:00


In [4]:
# ============================================
# VERIFY OPTIMIZATION ENVIRONMENT
# ============================================

import torch
import ultralytics
import torch_pruning as tp
import importlib.metadata

print("========== ENVIRONMENT ==========")
print("PyTorch       :", torch.__version__)
print("Ultralytics   :", ultralytics.__version__)
print("Torch-Pruning :", importlib.metadata.version("torch-pruning"))
print("CUDA          :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))
    print("CUDA version  :", torch.version.cuda)

print("=================================")

========== ENVIRONMENT ==========
PyTorch       : 2.11.0+cu128
Ultralytics   : 8.4.27
Torch-Pruning : 1.6.0
CUDA          : True
GPU           : Tesla T4
CUDA version  : 12.8


In [5]:
# ============================================
# UPLOAD ORIGINAL BASELINE MODEL
# ============================================

from google.colab import files
import os

print("Select your ORIGINAL best.pt file")
uploaded = files.upload()

uploaded_name = list(uploaded.keys())[0]

# Keep a clear, fixed name inside Colab
baseline_model_path = "/content/baseline_best.pt"

os.rename(f"/content/{uploaded_name}", baseline_model_path)

print("\nModel uploaded successfully!")
print("Path:", baseline_model_path)
print("Size:", os.path.getsize(baseline_model_path) / (1024 * 1024), "MB")

Select your ORIGINAL best.pt file


Saving best.pt to best.pt

Model uploaded successfully!
Path: /content/baseline_best.pt
Size: 49.606889724731445 MB


In [6]:
# ============================================
# LOAD BASELINE MODEL
# ============================================

from ultralytics import YOLO
import torch

MODEL_PATH = "/content/baseline_best.pt"

model = YOLO(MODEL_PATH)

print("========== BASELINE MODEL ==========")
print("Model loaded successfully")
print("Device:", "CUDA (T4)" if torch.cuda.is_available() else "CPU")

print("\nModel architecture:")
print(model.model)

print("\n====================================")

========== BASELINE MODEL ==========
Model loaded successfully
Device: CUDA (T4)

Model architecture:
DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(48, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(96, eps=0.00

In [7]:
# ============================================
# BASELINE MODEL COMPLEXITY
# ============================================

import torch_pruning as tp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_torch = model.model.to(device)
model_torch.eval()

# 640x640 input — same resolution as our original baseline
example_inputs = torch.randn(1, 3, 640, 640).to(device)

# Count parameters and MACs
macs, params = tp.utils.count_ops_and_params(
    model_torch,
    example_inputs
)

print("========== T4 BASELINE COMPLEXITY ==========")
print(f"Parameters : {params:,}")
print(f"Parameters : {params / 1e6:.3f} M")
print(f"MACs       : {macs:,.0f}")
print(f"MACs       : {macs / 1e9:.3f} G")
print(f"GFLOPs*    : {(macs * 2) / 1e9:.3f} G")
print("============================================")
print("* GFLOPs shown here use 2 FLOPs per MAC.")

========== T4 BASELINE COMPLEXITY ==========
Parameters : 25,856,899
Parameters : 25.857 M
MACs       : 39,439,458,000
MACs       : 39.439 G
GFLOPs*    : 78.879 G
* GFLOPs shown here use 2 FLOPs per MAC.


In [8]:
# ============================================
# T4 BASELINE INFERENCE SPEED
# ============================================

import time

# Warm-up
for _ in range(20):
    _ = model.predict(
        source=example_inputs,
        imgsz=640,
        device=0,
        verbose=False
    )

# Timed inference
torch.cuda.synchronize()
start = time.perf_counter()

for _ in range(100):
    _ = model.predict(
        source=example_inputs,
        imgsz=640,
        device=0,
        verbose=False
    )

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

avg_time = elapsed / 100
fps = 1 / avg_time

print("========== T4 BASELINE SPEED ==========")
print(f"Average inference time : {avg_time * 1000:.2f} ms/image")
print(f"FPS                    : {fps:.2f}")
print("========================================")

WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.068385601043701. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.068385601043701. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.068385601043701. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.068385601043701. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.068385601043701. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.068385601043701. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.068385601043701. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.068385601043701. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normali

In [9]:
!pip install -q torch-pruning

In [10]:
import importlib.metadata

print("Torch-Pruning version:", importlib.metadata.version("torch-pruning"))

import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch-Pruning version: 1.6.0
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [14]:
from ultralytics import YOLO
import torch

MODEL_PATH = "/content/baseline_best.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = YOLO(MODEL_PATH)

print("Model loaded successfully")
print("Device:", device)
print("Number of classes:", model.model.nc)
print("Number of layers:", len(model.model.model))

# Show the main architecture
print(model.model)

Model loaded successfully
Device: cuda
Number of classes: 1
Number of layers: 23
DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(48, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, aff

In [13]:
import os

print("Files in /content:")
for f in os.listdir("/content"):
    print(f)

Files in /content:
.config
baseline_best.pt
sample_data


In [15]:
import torch
import torch_pruning as tp

# Total parameters
total_params = sum(p.numel() for p in model.model.parameters())

print("========== BASELINE MODEL CHECK ==========")
print(f"Total parameters: {total_params:,}")
print(f"Total parameters (M): {total_params / 1e6:.3f} M")

print("\n========== PRUNABLE C2f LAYERS ==========")

c2f_layers = []

for name, module in model.model.named_modules():
    if module.__class__.__name__ == "C2f":
        c2f_layers.append((name, module))
        print(
            f"{name} | "
            f"input channels: {module.cv1.conv.in_channels} | "
            f"output channels: {module.cv2.conv.out_channels}"
        )

print(f"\nTotal C2f layers found: {len(c2f_layers)}")

print("\n========== DETECTION LAYER ==========")

for name, module in model.model.named_modules():
    if module.__class__.__name__ == "Detect":
        print(f"Detect layer found: {name}")
        print(module)

========== BASELINE MODEL CHECK ==========
Total parameters: 25,856,899
Total parameters (M): 25.857 M

========== PRUNABLE C2f LAYERS ==========
model.2 | input channels: 96 | output channels: 96
model.4 | input channels: 192 | output channels: 192
model.6 | input channels: 384 | output channels: 384
model.8 | input channels: 576 | output channels: 576
model.12 | input channels: 960 | output channels: 384
model.15 | input channels: 576 | output channels: 192
model.18 | input channels: 576 | output channels: 384
model.21 | input channels: 960 | output channels: 576

Total C2f layers found: 8

========== DETECTION LAYER ==========
Detect layer found: model.22
Detect(
  (cv2): ModuleList(
    (0): Sequential(
      (0): Conv(
        (conv): Conv2d(192, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): C